# Kalp Hastalığı Tahmini — Makine Öğrenmesi Ödevi

## Problem: İkili Sınıflandırma (Binary Classification)

**İkili sınıflandırma**, her örneği iki sınıftan birine ayıran bir makine öğrenmesi görevidir. Bu ödevde:

- **Sınıf 0 (No):** Kişide kalp hastalığı **yok**
- **Sınıf 1 (Yes):** Kişide kalp hastalığı **var**

Amaç: Yaş, tansiyon, kolesterol gibi özelliklere bakarak kişinin kalp hastalığı olup olmadığını tahmin etmek.

## Veri Seti

- **Kaynak:** [Kaggle — Heart Disease](https://www.kaggle.com/datasets/oktayrdeki/heart-disease)
- **Dosya:** `data/heart_disease.csv`

## Kullanılacak 5 Model (Kısa Tanım)

| Model | Ne yapar? |
|-------|-----------|
| **Logistic Regression** | Özelliklerin doğrusal birleşimine dayanarak olasılık hesaplar. |
| **Random Forest** | Birçok karar ağacının oylamasıyla tahmin üretir. |
| **SVM** | Veriyi en iyi ayıran sınırı (hyperplane) bulmaya çalışır. |
| **Gradient Boosting** | Hataları düzelten ardışık zayıf modeller birleştirir. |
| **KNN** | En yakın k komşunun etiketine bakarak sınıflandırır. |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

RANDOM_STATE = 42
TARGET_COL = 'Heart Disease Status'
DATA_PATH = 'data/heart_disease.csv'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline


## 1. Veriyi Yükleme ve İlk İnceleme

**Veri seti (dataset):** Modelin öğreneceği geçmiş kayıtların tablosudur.
- **Satır (örnek/sample):** Bir hastanın kaydı
- **Sütun (özellik/feature):** Yaş, kolesterol vb.
- **Hedef (target/label):** Tahmin etmek istediğimiz sütun: `Heart Disease Status`


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Satır sayısı: {len(df):,}')
print(f'Sütun sayısı: {len(df.columns)}')
print()
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


### Hedef Değişken Dağılımı

Modelin tahmin edeceği sütunun dağılımına bakıyoruz. Sınıflar **dengesiz** ise (örneğin %90 hayır, %10 evet) **accuracy** yanıltıcı olabilir; bu yüzden F1 ve ROC-AUC metriklerine de bakacağız.


In [ ]:
target_counts = df[TARGET_COL].value_counts()
print(target_counts)
print()
print('Yüzde dağılım:')
print((target_counts / len(df) * 100).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='black')
ax.set_title('Hedef Değişken: Kalp Hastalığı Durumu')
ax.set_xlabel('Durum')
ax.set_ylabel('Kişi Sayısı')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Keşifsel Veri Analizi (EDA)

**EDA (Exploratory Data Analysis):** Model kurmadan önce veriyi grafikler ve özet istatistiklerle tanımak.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Eksik Sayı': missing, 'Eksik %': missing_pct})
missing_df = missing_df[missing_df['Eksik Sayı'] > 0].sort_values('Eksik Sayı', ascending=False)
print('Eksik değer bulunan sütunlar:')
missing_df


### Eksik Değer Stratejisi

- **Sayısal sütunlar:** **Medyan** ile doldurma — aykırı değerlere ortalamadan daha dayanıklı.
- **Kategorik sütunlar:** **En sık görülen değer (mod)** ile doldurma.

Bu işlemler `SimpleImputer` ile yapılacak; doldurma istatistikleri **yalnızca eğitim setinden** öğrenilecek (veri sızıntısı önlenir).


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != TARGET_COL]

print('Sayısal özellikler:', len(numeric_cols))
print(numeric_cols)
print()
print('Kategorik özellikler:', len(categorical_cols))
print(categorical_cols)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols[:9]):
    axes[i].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
plt.suptitle('Sayısal Özelliklerin Dağılımı (Histogram)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Sayısal Özellikler Korelasyon Isı Haritası')
plt.tight_layout()
plt.show()


In [ ]:
cat_sample = categorical_cols[:4]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
for i, col in enumerate(cat_sample):
    df[col].value_counts().plot(kind='bar', ax=axes[i], edgecolor='black')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Örnek Kategorik Özellik Dağılımları')
plt.tight_layout()
plt.show()


## 3. Ön İşleme (Preprocessing)

### Train / Test Ayrımı

Veriyi ikiye ayırıyoruz:
- **Eğitim seti (%80):** Model buradan öğrenir.
- **Test seti (%20):** Model **hiç görmediği** veriyle ölçülür.

`stratify=y`: Her iki sette de Yes/No oranı ana veriyle aynı kalır.

### Hedef Kodlama (Label Encoding)

- `No` → 0
- `Yes` → 1


In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({'No': 0, 'Yes': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Eğitim: {len(X_train):,} örnek')
print(f'Test: {len(X_test):,} örnek')
print(f'Eğitimde Yes oranı: {y_train.mean():.2%}')
print(f'Teste Yes oranı: {y_test.mean():.2%}')


### Ön İşleme Pipeline Bileşenleri

| Bileşen | Açıklama |
|---------|----------|
| **SimpleImputer** | Eksik değerleri doldurur. |
| **OneHotEncoder** | Kategorik değerleri 0/1 sütunlarına çevirir. |
| **StandardScaler** | Sayısal özellikleri ortalama 0, std 1 yapar. SVM ve KNN için kritik. |
| **ColumnTransformer** | Sayısal ve kategorik sütunlara farklı işlem uygular. |
| **Pipeline** | Ön işleme + model tek akışta; test verisi eğitim istatistikleriyle dönüştürülür. |


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)


## 4. Modellerin Eğitimi

Her model aynı ön işleme adımlarını kullanır. Böylece karşılaştırma adil olur.

### Model Özetleri

1. **Logistic Regression:** Hızlı, yorumlanabilir; doğrusal ilişkilerde güçlü.
2. **Random Forest:** Ağaç topluluğu; karmaşık ilişkileri yakalayabilir.
3. **SVM:** Yüksek boyutta iyi çalışabilir; ölçekleme şart.
4. **Gradient Boosting:** Genelde güçlü performans; eğitim süresi daha uzun olabilir.
5. **KNN:** Basit; k ve ölçekleme seçimi önemli (burada k=5).


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

fitted_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    print(f'Eğitiliyor: {name}...')
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    predictions[name] = pipe.predict(X_test)
    probabilities[name] = pipe.predict_proba(X_test)[:, 1]

print('\nTüm modeller eğitildi.')


## 5. Değerlendirme Metrikleri

Modeli test setinde ölçüyoruz. Aşağıdaki kavramlar **pozitif sınıf = 1 (Yes, hastalık var)** için tanımlanır.

### Karışıklık Matrisi (Confusion Matrix)

| | Tahmin: 0 (No) | Tahmin: 1 (Yes) |
|---|----------------|-----------------|
| **Gerçek: 0** | TN (True Negative) | FP (False Positive) |
| **Gerçek: 1** | FN (False Negative) | TP (True Positive) |

- **TP:** Hastalığı var dedik, gerçekten vardı.
- **TN:** Yok dedik, gerçekten yoktu.
- **FP:** Var dedik, yoktu (yanlış alarm).
- **FN:** Yok dedik, vardı (**tehlikeli** — hastayı kaçırdık).

### Accuracy (Doğruluk)

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Tüm doğru tahminlerin oranı. Sınıflar dengesizse yanıltıcı olabilir.

### Precision (Kesinlik)

$$\text{Precision} = \frac{TP}{TP + FP}$$

"Evet hastalık var" dediğimiz kişilerin ne kadarı gerçekten hasta?

### Recall (Duyarlılık / Geri Çağırma)

$$\text{Recall} = \frac{TP}{TP + FN}$$

Gerçek hastaların ne kadarını yakaladık? Sağlıkta FN maliyeti yüksek olduğu için önemli.

### F1-Score

Precision ve Recall'un harmonik ortalaması:

$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

### ROC-AUC

**ROC eğrisi:** Farklı eşik değerlerinde True Positive Rate vs False Positive Rate.
**AUC:** Eğrinin altındaki alan (0.5 = rastgele, 1.0 = mükemmel). Sınıf dengesizliğinde accuracy'den daha güvenilir olabilir.


In [ ]:
results = []

for name in models.keys():
    y_pred = predictions[name]
    y_prob = probabilities[name]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df_rounded = results_df.round(4)
results_df_rounded


In [ ]:
# Örnek: en iyi ROC-AUC'ye sahip modelin karışıklık matrisi
best_by_auc = results_df['ROC-AUC'].idxmax()
y_pred_best = predictions[best_by_auc]
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tahmin: No (0)', 'Tahmin: Yes (1)'],
            yticklabels=['Gerçek: No (0)', 'Gerçek: Yes (1)'])
plt.title(f'Karışıklık Matrisi — {best_by_auc}')
plt.ylabel('Gerçek Değer')
plt.xlabel('Tahmin')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print()
print(classification_report(y_test, y_pred_best, target_names=['No (0)', 'Yes (1)']))


In [ ]:
# Tüm modeller için karışıklık matrisleri
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, name in enumerate(models.keys()):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    axes[i].set_title(name)
    axes[i].set_xlabel('Tahmin')
    axes[i].set_ylabel('Gerçek')

axes[-1].axis('off')
plt.suptitle('Tüm Modeller — Karışıklık Matrisleri', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 6. Model Karşılaştırması

Tüm metrikleri tek tabloda topluyoruz ve görselleştiriyoruz.


In [ ]:
display(results_df_rounded)

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
plot_df = results_df[metrics_to_plot].reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_to_plot))
width = 0.15
colors = sns.color_palette('husl', len(models))

for i, (_, row) in enumerate(plot_df.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    ax.bar(x + i * width, values, width, label=row['Model'], color=colors[i], edgecolor='black')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Skor')
ax.set_title('Model Karşılaştırması — Test Seti Metrikleri')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
for name in models.keys():
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Rastgele (AUC=0.5)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Eğrileri — Model Karşılaştırması')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


## 7. En İyi Model Seçimi

**Seçim kuralı:**
1. **Birincil kriter:** En yüksek **ROC-AUC**
2. **Eşitlik durumunda:** En yüksek **F1-Score**
3. **Sağlık notu:** Kaçırılan hasta (FN) maliyetli olduğu için yüksek **Recall** da önemlidir; seçilen modelin recall değeri de yorumlanacaktır.


In [ ]:
best_roc = results_df['ROC-AUC'].max()
candidates = results_df[results_df['ROC-AUC'] == best_roc]

if len(candidates) == 1:
    best_model = candidates.index[0]
else:
    best_model = candidates['F1-Score'].idxmax()

best_row = results_df.loc[best_model]

print('=' * 50)
print(f'EN İYİ MODEL: {best_model}')
print('=' * 50)
print(best_row.round(4))
print()
print('Gerekçe:')
print(f'- ROC-AUC = {best_row["ROC-AUC"]:.4f} (birincil kriter)')
print(f'- F1-Score = {best_row["F1-Score"]:.4f}')
print(f'- Recall = {best_row["Recall"]:.4f} (hastaların yakalanma oranı)')
print(f'- Precision = {best_row["Precision"]:.4f}')
print(f'- Accuracy = {best_row["Accuracy"]:.4f}')


In [ ]:
ranking = results_df.sort_values(['ROC-AUC', 'F1-Score'], ascending=False)
print('Genel sıralama (ROC-AUC, sonra F1):')
print()
ranking.round(4)


## 8. Özet ve Sonuç

Bu ödevde şunları yaptık:

1. **Veri keşfi:** Hedef dağılımı, eksik değerler, sayısal/kategorik özellikler incelendi.
2. **Ön işleme:** Eksik doldurma, one-hot encoding, ölçekleme ve train/test ayrımı uygulandı.
3. **5 model eğitildi:** Logistic Regression, Random Forest, SVM, Gradient Boosting, KNN.
4. **Metrikler hesaplandı:** Accuracy, Precision, Recall, F1, ROC-AUC ve karışıklık matrisi.
5. **Karşılaştırma:** Modeller tablo ve grafiklerle kıyaslandı; ROC-AUC ve F1'e göre en iyi model seçildi.

**Öğrenilen temel kavramlar:** ikili sınıflandırma, train/test split, pipeline, imputation, encoding, scaling, confusion matrix, precision, recall, F1, ROC-AUC.

> **Not:** Bu bir eğitim ödevidir; gerçek klinik kararlar için doktor değerlendirmesi esastır.
